# StudyMate: AI Learning Assistant
### NLP Mini Project — Weak Topic Detection & Personalized Learning

**Reference dataset:** Microsoft MS MARCO QA v2.1. It contains 808,731 training, 101,093 validation and 101,092 test queries (>1 million total). The notebook streams a 20,000-row sample from Hugging Face rather than packaging the multi-GB dataset.


In [1]:
!pip -q install datasets sentence-transformers scikit-learn pandas numpy matplotlib seaborn



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re, itertools
import numpy as np, pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

## 1. Load MS MARCO sample (streaming)
The full dataset is intentionally not bundled (it is ~2 GB). We stream only the first 20,000 training records so the demo stays light, reproducible and works offline-friendly.

In [3]:
N = 20000
ds = load_dataset('microsoft/ms_marco', 'v2.1', split='train', streaming=True)
rows = []
for x in itertools.islice(ds, N):
    answers = x.get('answers', [])
    ans = answers[0] if answers else ''
    rows.append((x['query'], ans))
df = pd.DataFrame(rows, columns=['question', 'answer'])
df = df[df.answer.str.len() > 10].reset_index(drop=True)
print('Usable sample:', len(df))
df.head()

Usable sample: 18353


,question,answer
0,)what was the immediate impact of the success ...,The immediate impact of the success of the man...
1,_________ justice is designed to repair the ha...,Restorative justice that fosters dialogue betw...
2,why did stalin want control of eastern europe,The reasons why Stalin wanted to control Easte...
3,why do nails get rusty,Nails rust in water because water allows the i...
4,depona ab,"Depona Ab is a library in Vilhelmina, Sweden."


## 2. Create answer-quality labels
For a mini-project experiment, we create weak/strong pairs automatically from the reference answer, then split them into train and test. The final academic version should replace these proxy labels with human-labelled student answers.

In [4]:
def weak_version(s):
    words = s.split()
    return ' '.join(words[:max(3, len(words)//3)])
def tail_version(s):
    words = s.split()
    return ' '.join(words[-max(3, len(words)//5):])
pairs = []
for _, r in df.head(10000).iterrows():
    if len(r.answer.split()) < 12:
        continue
    pairs.append((r.answer, r.answer, 1))
    pairs.append((weak_version(r.answer), r.answer, 0))
    pairs.append((tail_version(r.answer), r.answer, 0))
pairs = pd.DataFrame(pairs, columns=['student_answer', 'reference_answer', 'label'])
pairs = pairs.sample(frac=1, random_state=42).reset_index(drop=True)
train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, stratify=pairs.label, random_state=42)
print(f'Train pairs: {len(train_pairs)} | Test pairs: {len(test_pairs)}')
train_pairs.head()

Train pairs: 7622 | Test pairs: 1906


,student_answer,reference_answer,label
1790,The pathologic conditions or disorders such as...,The pathologic conditions or disorders such as...,1
3138,of the pelvic floor.,"Interstitial Cystitis (IC), is a pelvic pain d...",0
7790,"A gemstone symbolize wealth, power, and beauty...","A gemstone symbolize wealth, power, and beauty...",1
6708,eligible for SSI.,"Yes,But up to age 18 who meet strict disabilit...",0
4251,"From $2,500 to $7,500 for setting up, $1,000 t...","From $2,500 to $7,500 for setting up, $1,000 t...",1


## 3. Baseline: TF-IDF cosine similarity (held-out test)

In [5]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=100000)
Xtr = tfidf.fit_transform(train_pairs.student_answer.tolist() + train_pairs.reference_answer.tolist())
Xte = tfidf.transform(test_pairs.student_answer.tolist() + test_pairs.reference_answer.tolist())
ntr, nte = len(train_pairs), len(test_pairs)
Atr, Btr = Xtr[:ntr], Xtr[ntr:]
Ate, Bte = Xte[:nte], Xte[nte:]
tr_scores = np.asarray(normalize(Atr).multiply(normalize(Btr)).sum(axis=1)).ravel()
te_scores = np.asarray(normalize(Ate).multiply(normalize(Bte)).sum(axis=1)).ravel()
baseline_thr = max((t for t in np.linspace(0.05, 0.95, 91)),
                   key=lambda t: f1_score(train_pairs.label, (tr_scores >= t).astype(int), zero_division=0))
baseline_pred = (te_scores >= baseline_thr).astype(int)
b_acc = accuracy_score(test_pairs.label, baseline_pred)
b_p, b_r, b_f1, _ = precision_recall_fscore_support(test_pairs.label, baseline_pred, average='binary', zero_division=0)
print(f'Baseline TF-IDF | chosen threshold: {baseline_thr:.2f}')
print(f'Test Accuracy: {b_acc:.4f} | Precision: {b_p:.4f} | Recall: {b_r:.4f} | F1: {b_f1:.4f}')

Baseline TF-IDF | chosen threshold: 0.91
Test Accuracy: 1.0000 | Precision: 1.0000 | Recall: 1.0000 | F1: 1.0000


## 4. Improved model: Sentence Transformer semantic similarity (held-out test)

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
train_emb_s = model.encode(train_pairs.student_answer.tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
train_emb_r = model.encode(train_pairs.reference_answer.tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
train_sem = np.sum(train_emb_s * train_emb_r, axis=1)
test_emb_s = model.encode(test_pairs.student_answer.tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
test_emb_r = model.encode(test_pairs.reference_answer.tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
sem_scores = np.sum(test_emb_s * test_emb_r, axis=1)
sem_thr = max((t for t in np.linspace(0.05, 0.95, 91)),
              key=lambda t: f1_score(train_pairs.label, (train_sem >= t).astype(int), zero_division=0))
improved_pred = (sem_scores >= sem_thr).astype(int)
i_acc = accuracy_score(test_pairs.label, improved_pred)
i_p, i_r, i_f1, _ = precision_recall_fscore_support(test_pairs.label, improved_pred, average='binary', zero_division=0)
print(f'Improved Semantic | chosen threshold: {sem_thr:.2f}')
print(f'Test Accuracy: {i_acc:.4f} | Precision: {i_p:.4f} | Recall: {i_r:.4f} | F1: {i_f1:.4f}')
print(f'Accuracy improvement over TF-IDF: {(i_acc - b_acc) * 100:.2f} percentage points')
cm = confusion_matrix(test_pairs.label, improved_pred)
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted Weak', 'Predicted Strong'])
ax.set_yticklabels(['Actual Weak', 'Actual Strong'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center')
ax.set_title('Confusion matrix - semantic model (test set)')
plt.show()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/120 [00:00<?, ?it/s]

Batches:   0%|          | 0/120 [00:00<?, ?it/s]

## 5. StudyMate weak-topic demo
Upload or paste study material. We extract important concepts, create a small topic-linked quiz, evaluate answers semantically, and recommend revision.

In [ ]:
material = '''Natural Language Processing enables computers to understand human language. Tokenization breaks text into tokens. TF-IDF represents terms using frequency and inverse document frequency. Word embeddings represent words as dense vectors. Word2Vec learns representations from local context. GloVe uses global word co-occurrence statistics. Transformers use self-attention to model relationships between tokens. BERT produces contextual representations and is useful for classification and question answering.'''
sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', material) if len(s.split()) >= 4]

def extract_topics(sentences, top_n=8, min_len=4):
    vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 1),
                          token_pattern=r'(?u)\b\w[\w-]*\w*\b')
    M = vec.fit_transform(sentences)
    terms = vec.get_feature_names_out()
    total = np.asarray(M.sum(axis=0)).ravel()
    generic = {'word', 'words', 'term', 'terms', 'text', 'use', 'uses', 'using', 'represent',
               'represents', 'representations', 'language', 'natural', 'processing', 'local',
               'context', 'dense', 'global', 'attention', 'statistics', 'relationships', 'tokens',
               'vectors', 'human', 'computers', 'question', 'answer', 'answering', 'classification',
               'frequency', 'inverse', 'understand', 'enables', 'learns', 'produces', 'useful',
               'model', 'models', 'breaks', 'document'}
    topics = []
    for r in range(M.shape[0]):
        row = M.getrow(r)
        if row.nnz == 0:
            continue
        cols, vals = row.indices, row.data
        order = sorted(range(len(cols)), key=lambda k: (-vals[k], -total[cols[k]]))
        chosen = None
        for k in order:
            t = terms[cols[k]]
            if len(t) >= min_len and t not in generic:
                chosen = t
                break
        if chosen is None or chosen in topics:
            continue
        topics.append(chosen)
        if len(topics) >= top_n:
            break
    return topics

topics = extract_topics(sentences)
print('Extracted topics:', topics)

Extracted topics: ['tokenization', 'tf-idf', 'embeddings', 'word2vec', 'glove', 'transformers', 'bert']


In [ ]:
quiz = []
for topic in topics[:5]:
    idx = [i for i, s in enumerate(sentences) if topic in s.lower()]
    if idx:
        quiz.append((topic, sentences[idx[0]]))

learner_answers = {
    'tokenization': 'Tokenization splits text into smaller units called tokens.',
    'tf-idf': 'It is some scoring thing for words.',
    'embeddings': 'Word embeddings represent words as dense numerical vectors.',
    'word2vec': 'Word2Vec learns word representations from nearby context words.',
    'glove': 'Something about statistics, I guess.',
    'transformers': 'Transformers use self-attention to model relationships between tokens.',
    'bert': 'BERT produces contextual representations useful for question answering.',
}

rows = []
for topic, ref in quiz:
    student = learner_answers.get(topic, 'I am not sure about this topic.')
    emb_s = model.encode(student, normalize_embeddings=True)
    emb_r = model.encode(ref, normalize_embeddings=True)
    score = float(np.dot(emb_s, emb_r)) * 100
    rows.append((topic, round(score, 1)))
profile = pd.DataFrame(rows, columns=['Topic', 'Score'])
profile['Status'] = profile.Score.apply(lambda x: 'Weak' if x < 55 else ('Needs Practice' if x < 75 else 'Strong'))
profile = profile.sort_values('Score').reset_index(drop=True)
print(profile.to_string(index=False))
print()
for _, r in profile.iterrows():
    if r.Status == 'Weak':
        print(f'- Revise "{r.Topic}" and retake a focused quiz.')
    elif r.Status == 'Needs Practice':
        print(f'- Practise "{r.Topic}" with additional exercises.')
    else:
        print(f'- "{r.Topic}" looks strong; keep it up.')
print()
print('Paraphrase check: a reworded answer should still count as correct.')
paraphrases = {
    'tokenization': 'Breaking text down into smaller pieces is called tokenizing.',
    'tf-idf': 'TF-IDF weights terms based on how often they appear in a document.',
    'word2vec': 'Word2Vec learns word embeddings from the surrounding context words.',
}
for topic, ref in quiz:
    if topic not in paraphrases:
        continue
    par = paraphrases[topic]
    emb_s = model.encode(par, normalize_embeddings=True)
    emb_r = model.encode(ref, normalize_embeddings=True)
    sem = float(np.dot(emb_s, emb_r)) * 100
    M_p = TfidfVectorizer(stop_words='english').fit_transform([par, ref])
    lex = float(normalize(M_p[0]).multiply(normalize(M_p[1])).sum()) * 100
    print(f'{topic:12} semantic={sem:5.1f}%   tf-idf lexical={lex:5.1f}%')


       Topic  Score Status
       glove   28.6   Weak
      tf-idf   31.0   Weak
tokenization   78.3 Strong
  embeddings   92.7 Strong
    word2vec   95.2 Strong

- Revise "glove" and retake a focused quiz.
- Revise "tf-idf" and retake a focused quiz.
- "tokenization" looks strong; keep it up.
- "embeddings" looks strong; keep it up.
- "word2vec" looks strong; keep it up.

Paraphrase check: a reworded answer should still count as correct.
tokenization semantic= 70.9%   tf-idf lexical= 11.5%
tf-idf       semantic= 85.4%   tf-idf lexical= 30.1%
word2vec     semantic= 89.3%   tf-idf lexical= 34.5%


## 6. Final interpretation
StudyMate does not report a made-up accuracy. The notebook evaluates the TF-IDF baseline and the transformer-based model on a held-out test split, with thresholds selected on the training split. Because the automatically generated proxy labels are truncated copies of the same reference, the task is easy for both methods; the semantic model's real advantage appears on reworded answers (see the paraphrase check above), which pure lexical matching fails at. For the final paper, replace the automatically generated proxy labels with a manually labelled student-answer test set to make the accuracy claim academically valid.